Adding Initial Data Loading Code

In [6]:
import pandas as pd
import os

# ── PATHS ──────────────────────────────────────────────────────────────
# Change to our project directory (so Python knows where to find files)
os.chdir("/Users/animesh/Documents/personal_git/melbourne-crime-liveability-dashboard")

# Define folder paths
RAW = "data/raw/"              # Where downloaded files are stored
PROCESSED = "data/processed/"  # Where we'll save cleaned data
os.makedirs(PROCESSED, exist_ok=True)  # Create processed folder if it doesn't exist

# ── CRIME DATA ─────────────────────────────────────────────────────────
# Load the Excel file
# sheet_name="Table 01" means: open the sheet named "Table 01"
# skiprows=0 means: don't skip any rows (we want everything)
print("📊 Loading crime data from Table 01...")
crime_raw = pd.read_excel(
    RAW + "crime_lga_2024.xlsx",
    sheet_name="Table 01",
    skiprows=0
)

# Print information about what we loaded
print(f"\n✓ Loaded {len(crime_raw)} rows")  # How many rows?
print(f"✓ Columns: {crime_raw.columns.tolist()}")  # What are the column names?
print(f"\n✓ First 5 rows:")
print(crime_raw.head())  # Show first 5 rows of data
print(f"\n✓ Data types:")
print(crime_raw.dtypes)  # What type is each column? (int, string, float, etc.)

📊 Loading crime data from Table 01...

✓ Loaded 870 rows
✓ Columns: ['Year', 'Year ending', 'Police Region', 'Local Government Area', 'Incidents Recorded', 'Rate per 100,000 population']

✓ First 5 rows:
   Year Year ending       Police Region Local Government Area  \
0  2024    December  1 North West Metro               Banyule   
1  2024    December  1 North West Metro              Brimbank   
2  2024    December  1 North West Metro               Darebin   
3  2024    December  1 North West Metro           Hobsons Bay   
4  2024    December  1 North West Metro                  Hume   

   Incidents Recorded  Rate per 100,000 population  
0                7590                  5702.170117  
1               14119                  7124.608800  
2               13982                  8771.785832  
3                5973                  6293.377882  
4               16427                  6074.756226  

✓ Data types:
Year                             int64
Year ending                     o

In [9]:
# ── MELBOURNE LGA LIST ──────────────────────────────────────────────────
# These are the 30 Greater Melbourne LGAs we care about
MELBOURNE_LGAS = [
    "Banyule", "Bayside", "Boroondara", "Brimbank", "Cardinia",
    "Casey", "Darebin", "Frankston", "Glen Eira", "Greater Dandenong",
    "Hobsons Bay", "Hume", "Knox", "Manningham", "Maribyrnong",
    "Maroondah", "Melbourne", "Melton", "Monash", "Moonee Valley",
    "Moreland", "Mornington Peninsula", "Nillumbik", "Port Phillip",
    "Stonnington", "Whitehorse", "Whittlesea", "Wyndham", "Yarra",
    "Yarra Ranges"
]

print(f"📍 Filtering to {len(MELBOURNE_LGAS)} Melbourne LGAs...")

# IMPORTANT: Strip leading/trailing spaces from LGA names in the Excel data
# This is why we didn't match before!
# We need to remove the spaces before comparing
crime_melbourne = crime_raw[
    crime_raw["Local Government Area"].str.strip().isin(MELBOURNE_LGAS)
].copy()

# Also strip spaces from the LGA column in our new dataframe
crime_melbourne["Local Government Area"] = crime_melbourne["Local Government Area"].str.strip()

print(f"✓ Filtered to {len(crime_melbourne)} rows")
print(f"✓ Unique LGAs: {crime_melbourne['Local Government Area'].nunique()}")
print(f"\n✓ LGAs found:")
print(sorted(crime_melbourne["Local Government Area"].unique()))

📍 Filtering to 30 Melbourne LGAs...
✓ Filtered to 290 rows
✓ Unique LGAs: 29

✓ LGAs found:
['Banyule', 'Bayside', 'Boroondara', 'Brimbank', 'Cardinia', 'Casey', 'Darebin', 'Frankston', 'Glen Eira', 'Greater Dandenong', 'Hobsons Bay', 'Hume', 'Knox', 'Manningham', 'Maribyrnong', 'Maroondah', 'Melbourne', 'Melton', 'Monash', 'Moonee Valley', 'Mornington Peninsula', 'Nillumbik', 'Port Phillip', 'Stonnington', 'Whitehorse', 'Whittlesea', 'Wyndham', 'Yarra', 'Yarra Ranges']


In [10]:
print("🧹 CLEANING DATA...\n")

# Start with a copy (so we don't accidentally change the original)
crime_clean = crime_melbourne.copy()

# ── RENAME COLUMNS ──────────────────────────────────────────────────────
print("Step 1: Renaming columns to be cleaner...")

# Convert all column names to lowercase and replace spaces with underscores
# Example: "Local Government Area" becomes "local_government_area"
crime_clean.columns = [col.lower().replace(" ", "_") for col in crime_clean.columns]

# Now rename specific columns to be shorter and clearer
crime_clean = crime_clean.rename(columns={
    "local_government_area": "lga_name",
    "incidents_recorded": "incidents",
    "rate_per_100,000_population": "rate_per_100k"
})

print(f"   ✓ New column names: {crime_clean.columns.tolist()}\n")

# ── FIX DATA TYPES ──────────────────────────────────────────────────────
print("Step 2: Converting data types...")

# Make sure each column has the right data type
# int = whole number (Year, incidents)
# float = decimal number (crime rate)
crime_clean["year"] = crime_clean["year"].astype(int)
crime_clean["incidents"] = crime_clean["incidents"].astype(int)
crime_clean["rate_per_100k"] = crime_clean["rate_per_100k"].astype(float)

print(f"   ✓ Year is now: {crime_clean['year'].dtype}")
print(f"   ✓ Incidents is now: {crime_clean['incidents'].dtype}")
print(f"   ✓ Rate is now: {crime_clean['rate_per_100k'].dtype}\n")

# ── STANDARDIZE LGA NAMES ──────────────────────────────────────────────
print("Step 3: Standardizing LGA names...")

# Make all LGA names UPPERCASE and remove extra spaces
# Example: "  Melbourne  " becomes "MELBOURNE"
crime_clean["lga_name"] = crime_clean["lga_name"].str.upper().str.strip()

print(f"   ✓ All LGA names now in UPPERCASE\n")

# ── REMOVE BAD DATA ──────────────────────────────────────────────────────
print("Step 4: Removing rows with zero incidents...")

# Store old count
old_count = len(crime_clean)

# Remove rows where incidents = 0 (no data for that LGA/year combination)
crime_clean = crime_clean[crime_clean["incidents"] > 0].copy()

# Show how many we removed
removed = old_count - len(crime_clean)
print(f"   ✓ Removed {removed} rows with 0 incidents")
print(f"   ✓ Rows remaining: {len(crime_clean)}\n")

# ── SHOW RESULTS ──────────────────────────────────────────────────────
print("="*50)
print("FINAL CLEANED DATA")
print("="*50)
print(f"\n✓ Total rows: {len(crime_clean)}")
print(f"✓ Total columns: {len(crime_clean.columns)}")
print(f"\n✓ Columns in cleaned data:")
print(crime_clean.columns.tolist())
print(f"\n✓ Data types:")
print(crime_clean.dtypes)
print(f"\n✓ Sample of cleaned data (first 10 rows):")
print(crime_clean.head(10))
print(f"\n✓ Summary statistics:")
print(crime_clean.describe())

🧹 CLEANING DATA...

Step 1: Renaming columns to be cleaner...
   ✓ New column names: ['year', 'year_ending', 'police_region', 'lga_name', 'incidents', 'rate_per_100k']

Step 2: Converting data types...
   ✓ Year is now: int64
   ✓ Incidents is now: int64
   ✓ Rate is now: float64

Step 3: Standardizing LGA names...
   ✓ All LGA names now in UPPERCASE

Step 4: Removing rows with zero incidents...
   ✓ Removed 0 rows with 0 incidents
   ✓ Rows remaining: 290

FINAL CLEANED DATA

✓ Total rows: 290
✓ Total columns: 6

✓ Columns in cleaned data:
['year', 'year_ending', 'police_region', 'lga_name', 'incidents', 'rate_per_100k']

✓ Data types:
year               int64
year_ending       object
police_region     object
lga_name          object
incidents          int64
rate_per_100k    float64
dtype: object

✓ Sample of cleaned data (first 10 rows):
    year year_ending       police_region       lga_name  incidents  \
0   2024    December  1 North West Metro        BANYULE       7590   
1   2024

In [11]:
print("💾 EXPORTING CLEANED DATA TO CSV\n")

# Save the cleaned data to a CSV file
# index=False means: don't save the row numbers as a column
output_path = PROCESSED + "crime_clean.csv"
crime_clean.to_csv(output_path, index=False)

print(f"✓ Successfully exported to: {output_path}")
print(f"\n✓ File details:")
print(f"   - Total rows: {len(crime_clean)}")
print(f"   - Total columns: {len(crime_clean.columns)}")
print(f"   - File size: {os.path.getsize(output_path) / 1024:.2f} KB")

# Verify the file exists
if os.path.exists(output_path):
    print(f"\n✓ ✅ CSV file verified and exists!")
else:
    print(f"\n❌ ERROR: File was not created!")

print(f"\n✓ Preview of exported data:")
print(crime_clean.head(15))

💾 EXPORTING CLEANED DATA TO CSV

✓ Successfully exported to: data/processed/crime_clean.csv

✓ File details:
   - Total rows: 290
   - Total columns: 6
   - File size: 17.25 KB

✓ ✅ CSV file verified and exists!

✓ Preview of exported data:
    year year_ending       police_region       lga_name  incidents  \
0   2024    December  1 North West Metro        BANYULE       7590   
1   2024    December  1 North West Metro       BRIMBANK      14119   
2   2024    December  1 North West Metro        DAREBIN      13982   
3   2024    December  1 North West Metro    HOBSONS BAY       5973   
4   2024    December  1 North West Metro           HUME      16427   
5   2024    December  1 North West Metro    MARIBYRNONG       9509   
6   2024    December  1 North West Metro      MELBOURNE      32840   
7   2024    December  1 North West Metro         MELTON      11471   
9   2024    December  1 North West Metro  MOONEE VALLEY       7905   
10  2024    December  1 North West Metro      NILLUMBIK    

In [ ]:
print("✅ VERIFYING CLEANED DATA\n")
print("="*50)

# Read the CSV file back to make sure it works
test_read = pd.read_csv(PROCESSED + "crime_clean.csv")

print("CSV Verification Results:")
print("="*50)

print(f"\n✓ CSV successfully read back from disk!")
print(f"\n✓ Data shape (rows, columns): {test_read.shape}")
print(f"✓ File size: {os.path.getsize(PROCESSED + 'crime_clean.csv') / 1024:.2f} KB")

print(f"\n✓ Column names ({len(test_read.columns)} columns):")
for col in test_read.columns:
    print(f"   - {col}")

print(f"\n✓ Data types:")
print(test_read.dtypes)

print(f"\n✓ Year range:")
print(f"   - From: {test_read['year'].min()}")
print(f"   - To: {test_read['year'].max()}")

print(f"\n✓ LGAs included ({test_read['lga_name'].nunique()} unique):")
print(sorted(test_read['lga_name'].unique()))

print(f"\n✓ Basic statistics:")
print(test_read[['incidents', 'rate_per_100k']].describe())

print(f"\n✓ Sample data (first 10 rows):")
print(test_read.head(10))

print(f"\n✓ Sample data (random 5 rows):")
print(test_read.sample(5))

print("\n" + "="*50)
print("🎉 PHASE 1 COMPLETE!")
print("="*50)
print(f"\n✅ our cleaned data is ready!")
print(f"✅ Location: {PROCESSED}crime_clean.csv")
print(f"✅ Ready for Phase 2: SQL Analysis")

✅ VERIFYING CLEANED DATA

CSV Verification Results:

✓ CSV successfully read back from disk!

✓ Data shape (rows, columns): (290, 6)
✓ File size: 17.25 KB

✓ Column names (6 columns):
   - year
   - year_ending
   - police_region
   - lga_name
   - incidents
   - rate_per_100k

✓ Data types:
year               int64
year_ending       object
police_region     object
lga_name          object
incidents          int64
rate_per_100k    float64
dtype: object

✓ Year range:
   - From: 2015
   - To: 2024

✓ LGAs included (29 unique):
['BANYULE', 'BAYSIDE', 'BOROONDARA', 'BRIMBANK', 'CARDINIA', 'CASEY', 'DAREBIN', 'FRANKSTON', 'GLEN EIRA', 'GREATER DANDENONG', 'HOBSONS BAY', 'HUME', 'KNOX', 'MANNINGHAM', 'MARIBYRNONG', 'MAROONDAH', 'MELBOURNE', 'MELTON', 'MONASH', 'MOONEE VALLEY', 'MORNINGTON PENINSULA', 'NILLUMBIK', 'PORT PHILLIP', 'STONNINGTON', 'WHITEHORSE', 'WHITTLESEA', 'WYNDHAM', 'YARRA', 'YARRA RANGES']

✓ Basic statistics:
          incidents  rate_per_100k
count    290.000000     290.0